# 05 - Demo resets for the coworker agent (SYNTHETIC data)

Admin-only resets on the private ERP/WMS Azure SQL database. It is reachable only from this VNet, so these run in Databricks.

| `action` | What it does |
|---|---|
| `reset_store_transfers` | For one style and destination market: cancels the agent's unshipped transfer orders (status `CREATED`; seeded `STO-H%` history is untouched), gives the reserved units back to the sending store or DC, and closes any open case so the next event opens a new case and Teams chat. Use it when a new case shows only DC options because earlier approvals used up the spare store stock. |
| `close_case` | Closes one agent case without a business decision (no ERP write-back): cancels its proposed drafts and pending follow-ups. |

`mode = preview` runs the change inside a transaction, prints what would change and **rolls back**. `mode = apply` commits. Every applied reset writes one row to `agent.action_log`. Decisions already in `erp.allocation_decision` are kept as history. Fabric mirroring picks up the change within a few minutes.

In [ ]:
dbutils.widgets.dropdown("action", "reset_store_transfers", ["reset_store_transfers", "close_case"], "1. action")
dbutils.widgets.dropdown("mode", "preview", ["preview", "apply"], "2. mode")
dbutils.widgets.text("style_id", "HM9443", "3. style (reset_store_transfers)")
dbutils.widgets.text("market_id", "US", "4. destination market (reset_store_transfers)")
dbutils.widgets.text("case_id", "", "5. case id (close_case)")
dbutils.widgets.text("sql_server", "", "a. SQL server FQDN")
dbutils.widgets.text("sql_database", "sqldb-appareliq-erpwms", "b. SQL database")
dbutils.widgets.text("service_credential", "appareliq-sql", "c. UC service credential")


## Connect

In [ ]:
import json, re
ACTION, MODE = dbutils.widgets.get("action"), dbutils.widgets.get("mode")
STYLE_ID, MARKET_ID = dbutils.widgets.get("style_id").strip().upper(), dbutils.widgets.get("market_id").strip().upper()
CASE_ID = dbutils.widgets.get("case_id").strip()
if ACTION == "reset_store_transfers":
    assert re.fullmatch(r"[A-Z0-9]{2,16}", STYLE_ID) and re.fullmatch(r"[A-Z]{2}", MARKET_ID), "check style_id / market_id"
else:
    assert re.fullmatch(r"[A-Za-z0-9_.\-]{3,40}", CASE_ID), "set case_id (e.g. C-...)"
server, database = dbutils.widgets.get("sql_server"), dbutils.widgets.get("sql_database")
assert server, "set the sql_server widget"
token = dbutils.credentials.getServiceCredentialsProvider(dbutils.widgets.get("service_credential")).get_token("https://database.windows.net/.default").token
url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;"
       "hostNameInCertificate=*.database.windows.net;loginTimeout=60")
props = spark._jvm.java.util.Properties(); props.setProperty("accessToken", token)


def run_in_transaction(sql, params):
    """Run one parameterised batch in a transaction; commit in apply mode, roll back in preview mode."""
    conn = spark._jvm.java.sql.DriverManager.getConnection(url, props)
    conn.setAutoCommit(False)
    out = []
    try:
        st = conn.prepareStatement(sql)
        for k, v in enumerate(params, start=1):
            st.setString(k, v)
        has_rs = st.execute()
        while True:
            if has_rs:
                rs = st.getResultSet(); m = rs.getMetaData(); n = m.getColumnCount()
                cols = [m.getColumnLabel(k + 1) for k in range(n)]
                while rs.next():
                    out.append({cols[k]: (None if rs.getObject(k + 1) is None else str(rs.getObject(k + 1))) for k in range(n)})
            elif st.getUpdateCount() == -1:
                break
            has_rs = st.getMoreResults()
        if MODE == "apply":
            conn.commit()
        else:
            conn.rollback()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()
    return out

RESULT = None
print(f"action={ACTION} mode={MODE}")


## `reset_store_transfers`: give spare store stock back for one style and market

In [ ]:
RESET_STORE_TRANSFERS_SQL = r'''
SET NOCOUNT ON;
SET XACT_ABORT ON;
DECLARE @style NVARCHAR(16) = ?, @market NVARCHAR(8) = ?;
DECLARE @lines TABLE ([sto_id] NVARCHAR(32), [line_no] INT, [sku_id] NVARCHAR(48), [from_location_id] NVARCHAR(32), [units] INT);

INSERT INTO @lines
SELECT s.[sto_id], s.[line_no], s.[sku_id], s.[from_location_id], s.[units]
FROM [erp].[stock_transfer_order] s WITH (UPDLOCK)
JOIN [erp].[product_sku] k ON k.[sku_id] = s.[sku_id]
JOIN [erp].[location] t ON t.[location_id] = s.[to_location_id]
WHERE k.[style_id] = @style AND t.[market_id] = @market AND s.[sto_status] = N'CREATED' AND s.[sto_id] NOT LIKE N'STO-H%';

UPDATE i SET [reserved_units] = CASE WHEN ISNULL(i.[reserved_units], 0) > r.[units] THEN i.[reserved_units] - r.[units] ELSE 0 END,
             [snapshot_ts] = SYSUTCDATETIME()
FROM [wms].[inventory_position] i
JOIN (SELECT [from_location_id], [sku_id], SUM([units]) AS [units] FROM @lines GROUP BY [from_location_id], [sku_id]) r
  ON r.[from_location_id] = i.[location_id] AND r.[sku_id] = i.[sku_id];

UPDATE s SET [sto_status] = N'CANCELLED'
FROM [erp].[stock_transfer_order] s JOIN @lines l ON l.[sto_id] = s.[sto_id] AND l.[line_no] = s.[line_no];

DECLARE @open TABLE ([case_id] NVARCHAR(40), [prev] NVARCHAR(24));
UPDATE [agent].[case]
SET [case_status] = N'CLOSED', [pending_decision] = NULL, [pending_option_id] = NULL, [pending_units] = NULL,
    [pending_approver_upn] = NULL, [pending_approver_person_id] = NULL, [pending_rationale] = NULL,
    [pending_reason_code] = NULL, [defer_until_utc] = NULL, [updated_at_utc] = SYSUTCDATETIME()
OUTPUT inserted.[case_id], deleted.[case_status] INTO @open
WHERE [style_id] = @style AND [market_id] = @market AND [case_status] IN (N'OPEN', N'AWAITING_APPROVAL', N'NEEDS_FINANCE', N'DEFERRED');
UPDATE d SET [draft_status] = N'CANCELLED' FROM [agent].[transfer_draft] d JOIN @open o ON o.[case_id] = d.[case_id] WHERE d.[draft_status] = N'PROPOSED';
UPDATE f SET [task_status] = N'CANCELLED', [completed_at_utc] = SYSUTCDATETIME()
FROM [agent].[follow_up] f JOIN (SELECT DISTINCT d.[proposal_id] AS [case_id] FROM [erp].[allocation_decision] d
                                 JOIN [erp].[stock_transfer_order] s ON s.[decision_id] = d.[decision_id]
                                 JOIN @lines l ON l.[sto_id] = s.[sto_id] AND l.[line_no] = s.[line_no]
                                 UNION SELECT [case_id] FROM @open) c ON c.[case_id] = f.[case_id]
WHERE f.[task_status] = N'PENDING';

INSERT INTO [agent].[action_log] ([action_id], [case_id], [tier], [action_type], [actor], [outcome], [detail_json])
VALUES (CONCAT(N'reset-sto:', @style, N'-', @market, N':', CONVERT(NVARCHAR(36), NEWID())), NULL, 1, N'DEMO_RESET_STORE_TRANSFERS',
        N'admin-demo-reset', N'DONE',
        (SELECT @style AS style_id, @market AS market_id,
                (SELECT [sto_id], [line_no], [sku_id], [from_location_id], [units] FROM @lines FOR JSON PATH) AS cancelled_lines,
                (SELECT [case_id], [prev] AS previous_status FROM @open FOR JSON PATH) AS closed_cases
         FOR JSON PATH, WITHOUT_ARRAY_WRAPPER));

SELECT N'cancelled_sto_line' AS [kind], CONCAT([sto_id], N'#', [line_no], N' ', [sku_id], N' from ', [from_location_id], N' x', [units]) AS [detail] FROM @lines
UNION ALL SELECT N'closed_case', CONCAT([case_id], N' (was ', [prev], N')') FROM @open
UNION ALL
SELECT N'store_free_units', CONCAT(i.[location_id], N' ', i.[sku_id], N': ', i.[on_hand_units] - ISNULL(i.[allocated_units], 0) - ISNULL(i.[reserved_units], 0))
FROM [wms].[inventory_position] i JOIN [erp].[location] l ON l.[location_id] = i.[location_id]
JOIN [erp].[product_sku] k ON k.[sku_id] = i.[sku_id]
WHERE l.[location_type] = N'STORE' AND l.[market_id] = @market AND k.[style_id] = @style
  AND i.[location_id] IN (SELECT [from_location_id] FROM @lines);
'''
if ACTION == "reset_store_transfers":
    RESULT = run_in_transaction(RESET_STORE_TRANSFERS_SQL, [STYLE_ID, MARKET_ID])
    for r in RESULT:
        print(f"{r['kind']:20s} {r['detail']}")
    if not RESULT:
        print("nothing to reset: no unshipped agent transfer orders or open cases for", STYLE_ID, MARKET_ID)


## `close_case`: close one agent case without a decision

In [ ]:
CLOSE_CASE_SQL = r'''
SET NOCOUNT ON;
SET XACT_ABORT ON;
DECLARE @case_id NVARCHAR(40) = ?, @prev NVARCHAR(24);
SELECT @prev = [case_status] FROM [agent].[case] WITH (UPDLOCK) WHERE [case_id] = @case_id;
IF @prev IS NULL THROW 50000, N'case not found', 1;
UPDATE [agent].[case]
SET [case_status] = N'CLOSED', [pending_decision] = NULL, [pending_option_id] = NULL, [pending_units] = NULL,
    [pending_approver_upn] = NULL, [pending_approver_person_id] = NULL, [pending_rationale] = NULL,
    [pending_reason_code] = NULL, [defer_until_utc] = NULL, [updated_at_utc] = SYSUTCDATETIME()
WHERE [case_id] = @case_id;
UPDATE [agent].[transfer_draft] SET [draft_status] = N'CANCELLED' WHERE [case_id] = @case_id AND [draft_status] = N'PROPOSED';
UPDATE [agent].[follow_up] SET [task_status] = N'CANCELLED', [completed_at_utc] = SYSUTCDATETIME()
WHERE [case_id] = @case_id AND [task_status] = N'PENDING';
INSERT INTO [agent].[action_log] ([action_id], [case_id], [tier], [action_type], [actor], [outcome], [detail_json])
VALUES (CONCAT(N'reset:', @case_id, N':', CONVERT(NVARCHAR(36), NEWID())), @case_id, 1, N'DEMO_RESET_CLOSE', N'admin-demo-reset',
        N'DONE', (SELECT @prev AS previous_status, N'admin demo reset; no ERP write-back' AS note FOR JSON PATH, WITHOUT_ARRAY_WRAPPER));
SELECT [case_id], [case_status], [updated_at_utc] FROM [agent].[case] WHERE [case_id] = @case_id;
'''
if ACTION == "close_case":
    RESULT = run_in_transaction(CLOSE_CASE_SQL, [CASE_ID])
    for r in RESULT:
        print(r)


## Result

In [ ]:
print("COMMITTED" if MODE == "apply" else "PREVIEW ONLY - rolled back, nothing changed")
dbutils.notebook.exit(json.dumps({"action": ACTION, "mode": MODE, "rows": RESULT})[:60000])
